In [2]:
import numpy as np
from bokeh.plotting import figure, show
from bokeh.layouts import gridplot
from bokeh.io import output_notebook
from scipy import signal

# Активируем вывод в ноутбук
output_notebook()

class WindowedLowPassFilter:
    def __init__(self, sampling_rate, cutoff_freq, filter_order, window_type='hamming'):
        self.sampling_rate = sampling_rate
        self.cutoff_freq = cutoff_freq
        self.filter_order = filter_order
        self.window_type = window_type
        self.PI = np.pi
        self.FC = cutoff_freq / (sampling_rate / 2)
        
    def design_filter(self):
        M = self.filter_order
        FC = self.FC
        PI = self.PI
        
        H = np.zeros(M + 1)
        
        for i in range(M + 1):
            if (i - M/2) == 0:
                H[i] = 2 * PI * FC
            else:
                H[i] = np.sin(2 * PI * FC * (i - M/2)) / (i - M/2)
            
            if self.window_type == 'hamming':
                H[i] = H[i] * (0.54 - 0.46 * np.cos(2 * PI * i / M))
            elif self.window_type == 'hann':
                H[i] = H[i] * (0.5 - 0.5 * np.cos(2 * PI * i / M))
            elif self.window_type == 'blackman':
                H[i] = H[i] * (0.42 - 0.5 * np.cos(2 * PI * i / M) + 
                               0.08 * np.cos(4 * PI * i / M))
        
        H = H / np.sum(H)
        self.coefficients = H
        return H
    
    def apply_filter(self, input_signal):
        M = self.filter_order
        H = self.coefficients
        N = len(input_signal)
        output_signal = np.zeros(N)
        
        for j in range(M, N):
            y_j = 0
            print ("777" + str(j))
            for i in range(M + 1):
                y_j += input_signal[j - i] * H[i]
            output_signal[j] = y_j
        
        return output_signal
    
    def get_frequency_response(self, n_points=1024):
        w, h = signal.freqz(self.coefficients, worN=n_points)
        frequencies = w * self.sampling_rate / (2 * np.pi)
        magnitude = np.abs(h)
        return frequencies, magnitude
    
    def get_window_function(self):
        M = self.filter_order
        
        if self.window_type == 'hamming':
            window = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(M + 1) / M)
        elif self.window_type == 'hann':
            window = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(M + 1) / M)
        elif self.window_type == 'blackman':
            window = (0.42 - 0.5 * np.cos(2 * np.pi * np.arange(M + 1) / M) + 
                     0.08 * np.cos(4 * np.pi * np.arange(M + 1) / M))
        else:
            window = np.ones(M + 1)
        
        return window

def generate_test_signal(length, sampling_rate, frequencies, amplitudes=None, noise_level=0.1):
    if amplitudes is None:
        amplitudes = [1.0] * len(frequencies)
    
    t = np.arange(length) / sampling_rate
    signal_clean = np.zeros(length)
    
    for freq, amp in zip(frequencies, amplitudes):
        signal_clean += amp * np.sin(2 * np.pi * freq * t)
    
    noise = noise_level * np.random.randn(length)
    signal_noisy = signal_clean + noise
    
    return signal_noisy, signal_clean, t

def compute_spectrum(signal, sampling_rate):
    N = len(signal)
    fft_result = np.fft.fft(signal)
    frequencies = np.fft.fftfreq(N, 1/sampling_rate)
    
    positive_freq_idx = frequencies >= 0
    frequencies = frequencies[positive_freq_idx]
    magnitude = np.abs(fft_result[positive_freq_idx]) / N * 2
    magnitude[0] /= 2
    
    return frequencies, magnitude

def create_filter_plots(original_signal, filtered_signal, t, sampling_rate, filter_obj):
    # Вычисление спектров
    freq_orig, mag_orig = compute_spectrum(original_signal, sampling_rate)
    freq_filt, mag_filt = compute_spectrum(filtered_signal, sampling_rate)
    
    # АЧХ фильтра
    freq_response, mag_response = filter_obj.get_frequency_response()
    
    # Оконная функция
    window = filter_obj.get_window_function()
    window_x = np.arange(len(window))
    
    # Создание графиков
    p1 = figure(title="Сигналы во временной области", width=600, height=300,
                x_axis_label='Время (с)', y_axis_label='Амплитуда')
    p1.line(t, original_signal, legend_label='Исходный сигнал', 
            line_color='blue', line_width=2)
    p1.line(t, filtered_signal, legend_label='Отфильтрованный сигнал', 
            line_color='red', line_width=2)
    p1.legend.location = "top_right"
    
    p2 = figure(title="Спектры сигналов", width=600, height=300,
                x_axis_label='Частота (Гц)', y_axis_label='Амплитуда')
    p2.line(freq_orig, mag_orig, legend_label='Исходный спектр', 
            line_color='blue', line_width=2)
    p2.line(freq_filt, mag_filt, legend_label='Отфильтрованный спектр', 
            line_color='red', line_width=2)
    p2.legend.location = "top_right"
    
    p3 = figure(title="Оконная функция", width=600, height=300,
                x_axis_label='Отсчеты', y_axis_label='Амплитуда')
    p3.line(window_x, window, line_color='green', line_width=2)
    p3.circle(window_x, window, size=5, color='green', alpha=0.7)
    
    p4 = figure(title="АЧХ фильтра", width=600, height=300,
                x_axis_label='Частота (Гц)', y_axis_label='Коэффициент передачи')
    p4.line(freq_response, mag_response, line_color='purple', line_width=2)
    
    p4.line([filter_obj.cutoff_freq, filter_obj.cutoff_freq], 
            [0, max(mag_response)], line_color='red', 
            line_dash='dashed', line_width=1, legend_label=f'f_cut = {filter_obj.cutoff_freq} Гц')
    p4.legend.location = "top_right"
    
    # Отображение графиков
    plot_grid = gridplot([[p1, p2], [p3, p4]])
    show(plot_grid)

# Основная функция для демонстрации
def demo_lowpass_filter(sampling_rate=1000, cutoff_freq=100, filter_order=100, 
                       window_type='hamming', signal_length=2000,
                       frequencies=[50, 150, 300], amplitudes=[1.0, 0.5, 0.3]):
    
    print("=" * 60)
    print("ФИЛЬТР НИЖНИХ ЧАСТОТ С ОКОННЫМИ ФУНКЦИЯМИ")
    print("=" * 60)
    
    # Генерация тестового сигнала
    test_signal, clean_signal, time = generate_test_signal(
        signal_length, sampling_rate, frequencies, amplitudes, noise_level=0.2
    )
    
    # Создание и проектирование фильтра
    lp_filter = WindowedLowPassFilter(sampling_rate, cutoff_freq, filter_order, window_type)
    coefficients = lp_filter.design_filter()
    
    # Применение фильтра
    filtered_signal = lp_filter.apply_filter(test_signal)
    
    # Создание и отображение графиков
    create_filter_plots(test_signal, filtered_signal, time, sampling_rate, lp_filter)
    
    # Вывод информации
    print(f"ПАРАМЕТРЫ ФИЛЬТРА:")
    print(f"  Частота дискретизации: {sampling_rate} Гц")
    print(f"  Частота среза: {cutoff_freq} Гц")
    print(f"  Порядок фильтра: {filter_order}")
    print(f"  Тип окна: {window_type}")
    print(f"  Нормированная частота среза: {lp_filter.FC:.4f}")
    print(f"  Сумма коэффициентов фильтра: {np.sum(coefficients):.6f}")
    print()
    print(f"ПАРАМЕТРЫ СИГНАЛА:")
    print(f"  Длина сигнала: {signal_length} отсчетов")
    print(f"  Компоненты: {[f'{f} Гц (A={a})' for f, a in zip(frequencies, amplitudes)]}")
    print()
    
    return test_signal, filtered_signal, lp_filter

# Запуск демонстрации
print("Запуск демонстрации фильтра...")
test_signal, filtered_signal, filter_obj = demo_lowpass_filter()


Loading BokehJS ...

Запуск демонстрации фильтра...
ФИЛЬТР НИЖНИХ ЧАСТОТ С ОКОННЫМИ ФУНКЦИЯМИ
777100
777101
777102
777103
777104
777105
777106
777107
777108
777109
777110
777111
777112
777113
777114
777115
777116
777117
777118
777119
777120
777121
777122
777123
777124
777125
777126
777127
777128
777129
777130
777131
777132
777133
777134
777135
777136
777137
777138
777139
777140
777141
777142
777143
777144
777145
777146
777147
777148
777149
777150
777151
777152
777153
777154
777155
777156
777157
777158
777159
777160
777161
777162
777163
777164
777165
777166
777167
777168
777169
777170
777171
777172
777173
777174
777175
777176
777177
777178
777179
777180
777181
777182
777183
777184
777185
777186
777187
777188
777189
777190
777191
777192
777193
777194
777195
777196
777197
777198
777199
777200
777201
777202
777203
777204
777205
777206
777207
777208
777209
777210
777211
777212
777213
777214
777215
777216
777217
777218
777219
777220
777221
777222
777223
777224
777225
777226
777227
777228
777229
777230
777231
777

ПАРАМЕТРЫ ФИЛЬТРА:
  Частота дискретизации: 1000 Гц
  Частота среза: 100 Гц
  Порядок фильтра: 100
  Тип окна: hamming
  Нормированная частота среза: 0.2000
  Сумма коэффициентов фильтра: 1.000000

ПАРАМЕТРЫ СИГНАЛА:
  Длина сигнала: 2000 отсчетов
  Компоненты: ['50 Гц (A=1.0)', '150 Гц (A=0.5)', '300 Гц (A=0.3)']



In [15]:
import numpy as np
from bokeh.plotting import figure, show
from bokeh.layouts import gridplot
from bokeh.io import output_notebook
from scipy import signal

# Активируем вывод графиков Bokeh в Jupyter Notebook
output_notebook()

class WindowedLowPassFilter:
    """
    Класс для создания и применения оконного фильтра нижних частот
    Сохраняет логику оригинальной BASIC-программы
    """
    
    def __init__(self, sampling_rate, cutoff_freq, filter_order, window_type='hamming'):
        """
        Инициализация параметров фильтра
        
        Parameters:
        sampling_rate - частота дискретизации в Гц
        cutoff_freq - граничная частота среза в Гц  
        filter_order - порядок фильтра (количество коэффициентов)
        window_type - тип оконной функции ('hamming', 'hann', 'blackman', 'rectangular')
        """
        self.sampling_rate = sampling_rate
        self.cutoff_freq = cutoff_freq
        self.filter_order = filter_order
        self.window_type = window_type
        self.PI = np.pi
        
        # Нормированная частота среза (относительно частоты Найквиста)
        # В оригинальной программе FC задавалась в диапазоне 0...0.5
        self.FC = cutoff_freq / (sampling_rate / 2)
        
    def design_filter(self):
        """
        Расчет весовых коэффициентов фильтра
        Соответствует строкам 240-290 оригинальной BASIC-программы
        """
        M = self.filter_order
        FC = self.FC
        PI = self.PI
        
        # Инициализация массива коэффициентов (аналог DIM H[100] в BASIC)
        H = np.zeros(M + 1)
        
        # Вычисление идеальной импульсной характеристики НЧ-фильтра
        # Соответствует строкам 250-290 в BASIC
        for i in range(M + 1):
            # Расчет идеальной импульсной характеристики (формула 16.4)
            if (i - M/2) == 0:
                # Особый случай для i = M/2 (избегаем деления на ноль)
                H[i] = 2 * PI * FC
            else:
                # Основная формула для импульсной характеристики
                H[i] = np.sin(2 * PI * FC * (i - M/2)) / (i - M/2)
            
            # Применение оконной функции для уменьшения эффекта Гиббса
            # Соответствует строке 280 в BASIC
            if self.window_type == 'hamming':
                # Окно Хэмминга (0.54 - 0.46*cos(...))
                H[i] = H[i] * (0.54 - 0.46 * np.cos(2 * PI * i / M))
            elif self.window_type == 'hann':
                # Окно Хэнна (0.5 - 0.5*cos(...))
                H[i] = H[i] * (0.5 - 0.5 * np.cos(2 * PI * i / M))
            elif self.window_type == 'blackman':
                # Окно Блэкмана
                H[i] = H[i] * (0.42 - 0.5 * np.cos(2 * PI * i / M) + 
                               0.08 * np.cos(4 * PI * i / M))
            # Прямоугольное окно (без изменений) - соответствует исходной программе
        
        # Нормировка коэффициентов для единичного усиления на нулевой частоте
        # Соответствует строкам 310-380 в BASIC
        H = H / np.sum(H)
        
        self.coefficients = H
        return H
    
    def apply_filter(self, input_signal):
        """
        Применение фильтра к входному сигналу методом свертки
        Соответствует строкам 400-450 оригинальной BASIC-программы
        """
        M = self.filter_order
        H = self.coefficients
        N = len(input_signal)
        
        # Инициализация выходного сигнала (аналог DIM Y[5000] в BASIC)
        output_signal = np.zeros(N)
        
        # Операция свертки - соответствует строкам 400-450 в BASIC
        # В BASIC: FOR J% = 100 TO 4999
        for j in range(M, N):
            y_j = 0
            # В BASIC: FOR I% = 0 TO 100
            for i in range(M + 1):
                # Свертка: Y[J%] = Y[J%] + X[J%-I%] * H[I%]
                y_j += input_signal[j - i] * H[i]
            output_signal[j] = y_j
        
        return output_signal
    
    def get_frequency_response(self, n_points=1024):
        """
        Расчет амплитудно-частотной характеристики (АЧХ) фильтра
        Показывает, как фильтр воздействует на разные частоты
        """
        # Используем scipy для расчета частотной характеристики
        w, h = signal.freqz(self.coefficients, worN=n_points)
        # Преобразование нормированных частот в герцы
        frequencies = w * self.sampling_rate / (2 * np.pi)
        magnitude = np.abs(h)  # Амплитуда коэффициента передачи
        return frequencies, magnitude
    
    def get_window_function(self):
        """
        Получение оконной функции для визуализации
        """
        M = self.filter_order
        
        # Расчет различных типов оконных функций
        if self.window_type == 'hamming':
            window = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(M + 1) / M)
        elif self.window_type == 'hann':
            window = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(M + 1) / M)
        elif self.window_type == 'blackman':
            window = (0.42 - 0.5 * np.cos(2 * np.pi * np.arange(M + 1) / M) + 
                     0.08 * np.cos(4 * np.pi * np.arange(M + 1) / M))
        else:  # rectangular
            window = np.ones(M + 1)
        
        return window


def generate_test_signal(length, sampling_rate, frequencies, amplitudes=None, noise_level=0.1):
    """
    Генерация тестового сигнала для демонстрации работы фильтра
    
    Parameters:
    length - количество отсчетов сигнала
    sampling_rate - частота дискретизации в Гц
    frequencies - список частот синусоидальных компонент
    amplitudes - список амплитуд компонент
    noise_level - уровень случайного шума
    """
    if amplitudes is None:
        amplitudes = [1.0] * len(frequencies)
    
    # Временная ось
    t = np.arange(length) / sampling_rate
    signal_clean = np.zeros(length)
    
    # Создание многочастотного сигнала как суммы синусоид
    for freq, amp in zip(frequencies, amplitudes):
        signal_clean += amp * np.sin(2 * np.pi * freq * t)
    
    # Добавление белого гауссовского шума для реалистичности
    noise = noise_level * np.random.randn(length)
    signal_noisy = signal_clean + noise
    
    return signal_noisy, signal_clean, t


def compute_spectrum(signal, sampling_rate):
    """
    Вычисление амплитудного спектра сигнала с помощью БПФ
    
    Parameters:
    signal - входной сигнал
    sampling_rate - частота дискретизации
    
    Returns:
    frequencies - массив частот в Гц
    magnitude - амплитудный спектр
    """
    N = len(signal)
    # Вычисление быстрого преобразования Фурье
    fft_result = np.fft.fft(signal)
    # Получение частотной оси
    frequencies = np.fft.fftfreq(N, 1/sampling_rate)
    
    # Берем только положительные частоты (односторонний спектр)
    positive_freq_idx = frequencies >= 0
    frequencies = frequencies[positive_freq_idx]
    # Нормировка амплитуды
    magnitude = np.abs(fft_result[positive_freq_idx]) / N * 2
    magnitude[0] /= 2  # DC компонента не удваивается
    
    return frequencies, magnitude


def create_filter_plots(original_signal, filtered_signal, t, sampling_rate, filter_obj):
    """
    Создание и отображение графиков для анализа работы фильтра
    
    Создает 4 графика:
    1. Сигналы во временной области (до и после фильтрации)
    2. Амплитудные спектры сигналов
    3. Оконная функция фильтра
    4. АЧХ фильтра
    """
    
    # Вычисление спектров исходного и отфильтрованного сигналов
    freq_orig, mag_orig = compute_spectrum(original_signal, sampling_rate)
    freq_filt, mag_filt = compute_spectrum(filtered_signal, sampling_rate)
    
    # Получение АЧХ фильтра
    freq_response, mag_response = filter_obj.get_frequency_response()
    
    # Получение оконной функции
    window = filter_obj.get_window_function()
    window_x = np.arange(len(window))
    
    # 1. График сигналов во временной области
    p1 = figure(title="Сигналы во временной области", width=600, height=300,
                x_axis_label='Время (с)', y_axis_label='Амплитуда')
    p1.line(t, original_signal, legend_label='Исходный сигнал', 
            line_color='blue', line_width=2)
    p1.line(t, filtered_signal, legend_label='Отфильтрованный сигнал', 
            line_color='red', line_width=2)
    p1.legend.location = "top_right"
    
    # 2. График амплитудных спектров
    p2 = figure(title="Спектры сигналов", width=600, height=300,
                x_axis_label='Частота (Гц)', y_axis_label='Амплитуда')
    p2.line(freq_orig, mag_orig, legend_label='Исходный спектр', 
            line_color='blue', line_width=2)
    p2.line(freq_filt, mag_filt, legend_label='Отфильтрованный спектр', 
            line_color='red', line_width=2)
    p2.legend.location = "top_right"
    
    # 3. График оконной функции
    p3 = figure(title="Оконная функция", width=600, height=300,
                x_axis_label='Отсчеты', y_axis_label='Амплитуда')
    p3.line(window_x, window, line_color='green', line_width=2)
    p3.circle(window_x, window, size=5, color='green', alpha=0.7)
    
    # 4. График АЧХ фильтра
    p4 = figure(title="АЧХ фильтра", width=600, height=300,
                x_axis_label='Частота (Гц)', y_axis_label='Коэффициент передачи')
    p4.line(freq_response, mag_response, line_color='purple', line_width=2)
    
    # Вертикальная линия для обозначения частоты среза
    p4.line([filter_obj.cutoff_freq, filter_obj.cutoff_freq], 
            [0, max(mag_response)], line_color='red', 
            line_dash='dashed', line_width=1, 
            legend_label=f'f_cut = {filter_obj.cutoff_freq} Гц')
    p4.legend.location = "top_right"
    
    # Создание сетки 2x2 и отображение всех графиков
    plot_grid = gridplot([[p1, p2], [p3, p4]])
    show(plot_grid)


def demo_lowpass_filter(sampling_rate=1000, cutoff_freq=200/2, filter_order=1000, 
                       window_type='hamming', signal_length=2000,
                       frequencies=[50, 150, 300], amplitudes=[1.0, 0.5, 0.3]):
    """
    Основная функция демонстрации работы фильтра нижних частот
    
    Parameters:
    sampling_rate - частота дискретизации в Гц (по умолчанию 1000 Гц)
    cutoff_freq - частота среза в Гц (по умолчанию 100 Гц)
    filter_order - порядок фильтра (по умолчанию 100)
    window_type - тип окна (по умолчанию 'hamming')
    signal_length - длина тестового сигнала в отсчетах
    frequencies - частоты компонент тестового сигнала
    amplitudes - амплитуды компонент тестового сигнала
    """
    
    print("=" * 60)
    print("ФИЛЬТР НИЖНИХ ЧАСТОТ С ОКОННЫМИ ФУНКЦИЯМИ")
    print("=" * 60)
    
    # Генерация тестового сигнала (аналог строки 220 в BASIC)
    test_signal, clean_signal, time = generate_test_signal(
        signal_length, sampling_rate, frequencies, amplitudes, noise_level=0.2
    )
    
    # Создание и проектирование фильтра
    lp_filter = WindowedLowPassFilter(sampling_rate, cutoff_freq, filter_order, window_type)
    coefficients = lp_filter.design_filter()
    
    # Применение фильтра к сигналу
    filtered_signal = lp_filter.apply_filter(test_signal)
    
    # Создание и отображение графиков для визуального анализа
    create_filter_plots(test_signal, filtered_signal, time, sampling_rate, lp_filter)
    
    # Вывод информации о параметрах для изучения
    print(f"ПАРАМЕТРЫ ФИЛЬТРА:")
    print(f"  Частота дискретизации: {sampling_rate} Гц")
    print(f"  Частота среза: {cutoff_freq} Гц")
    print(f"  Порядок фильтра: {filter_order}")
    print(f"  Тип окна: {window_type}")
    print(f"  Нормированная частота среза: {lp_filter.FC:.4f}")
    print(f"  Сумма коэффициентов фильтра: {np.sum(coefficients):.6f}")
    print()
    print(f"ПАРАМЕТРЫ СИГНАЛА:")
    print(f"  Длина сигнала: {signal_length} отсчетов")
    print(f"  Компоненты: {[f'{f} Гц (A={a})' for f, a in zip(frequencies, amplitudes)]}")
    print()
    
    return test_signal, filtered_signal, lp_filter


# Запуск демонстрации работы фильтра
if __name__ == "__main__":
    # Этот блок выполнится только при прямом запуске скрипта
    # В Jupyter Notebook можно вызывать функцию demo_lowpass_filter() напрямую
    print("Запуск демонстрации фильтра нижних частот...")
    test_signal, filtered_signal, filter_obj = demo_lowpass_filter()

Loading BokehJS ...

Запуск демонстрации фильтра нижних частот...
ФИЛЬТР НИЖНИХ ЧАСТОТ С ОКОННЫМИ ФУНКЦИЯМИ


ПАРАМЕТРЫ ФИЛЬТРА:
  Частота дискретизации: 1000 Гц
  Частота среза: 100.0 Гц
  Порядок фильтра: 1000
  Тип окна: hamming
  Нормированная частота среза: 0.2000
  Сумма коэффициентов фильтра: 1.000000

ПАРАМЕТРЫ СИГНАЛА:
  Длина сигнала: 2000 отсчетов
  Компоненты: ['50 Гц (A=1.0)', '150 Гц (A=0.5)', '300 Гц (A=0.3)']



In [1]:
# Импорт необходимых библиотек
# numpy - для математических операций и работы с массивами
import numpy as np
# Bokeh - для создания интерактивных графиков
from bokeh.plotting import figure, show  # figure - создание графиков, show - отображение
from bokeh.layouts import gridplot       # gridplot - компоновка графиков в сетку
from bokeh.io import output_notebook     # output_notebook - вывод в Jupyter Notebook
# scipy.signal - для сигнальной обработки (расчет АЧХ)
from scipy import signal

# Активируем вывод графиков Bokeh непосредственно в Jupyter Notebook
# Это позволяет отображать графики прямо в ячейках ноутбука
output_notebook()

class WindowedLowPassFilter:
    """
    Класс для создания и применения оконного КИХ-фильтра нижних частот
    (КИХ = Конечная Импульсная Характеристика)
    
    Основные принципы:
    - Фильтр работает методом свертки входного сигнала с импульсной характеристикой
    - Используется метод окон для уменьшения эффекта Гиббса (пульсации в АЧХ)
    - Сохраняет логику оригинальной BASIC-программы 16.1
    
    Математическая основа:
    Импульсная характеристика идеального НЧ-фильтра: h[n] = sin(2π·fc·n) / (π·n)
    где fc - нормированная частота среза
    """
    
    def __init__(self, sampling_rate, cutoff_freq, filter_order, window_type='hamming'):
        """
        Конструктор класса - инициализирует основные параметры фильтра
        
        Параметры:
        ----------
        sampling_rate : int
            Частота дискретизации в Герцах (количество отсчетов в секунду)
            Определяет максимальную частоту по теореме Найквиста (f_nyquist = sampling_rate/2)
            
        cutoff_freq : float
            Граничная частота среза в Герцах. Частоты выше этой будут ослабляться фильтром
            
        filter_order : int
            Порядок фильтра (количество коэффициентов минус 1)
            Определяет крутизну склона АЧХ и вычислительную сложность
            Чем выше порядок - тем резче переходная полоса, но больше задержка
            
        window_type : str, optional
            Тип оконной функции для уменьшения эффекта Гиббса, по умолчанию 'hamming'
            Доступные варианты: 'hamming', 'hann', 'blackman', 'rectangular'
            Окна отличаются компромиссом между шириной главного лепестка и уровнем боковых лепестков
        """
        # Сохраняем параметры фильтра как атрибуты объекта
        self.sampling_rate = sampling_rate  # Частота дискретизации в Гц
        self.cutoff_freq = cutoff_freq      # Частота среза в Гц
        self.filter_order = filter_order    # Порядок фильтра
        self.window_type = window_type      # Тип оконной функции
        self.PI = np.pi                     # Константа π для математических расчетов
        
        # Нормированная частота среза (относительно частоты Найквиста)
        # В цифровой обработке частоты обычно нормируются к половине частоты дискретизации
        # В оригинальной BASIC-программе FC задавалась в диапазоне 0...0.5
        # Формула: FC = f_cutoff / (f_sampling / 2) = 2 * f_cutoff / f_sampling
        self.FC = cutoff_freq / (sampling_rate / 2)
        
    def design_filter(self):
        """
        Расчет весовых коэффициентов фильтра (импульсной характеристики)
        Соответствует строкам 240-290 оригинальной BASIC-программы
        
        Математическая основа:
        - Сначала рассчитывается идеальная импульсная характеристика НЧ-фильтра
        - Затем применяется оконная функция для уменьшения эффекта Гиббса
        - Наконец, коэффициенты нормируются для единичного усиления 
        
        Возвращает:
        -----------
        H : numpy.array
            Массив коэффициентов фильтра (импульсная характеристика)
        """
        # Локальные переменные для удобства чтения кода
        M = self.filter_order  # Порядок фильтра
        FC = self.FC           # Нормированная частота среза
        PI = self.PI           # Число π
        
        # Инициализация массива коэффициентов (аналог DIM H[100] в BASIC)
        # Размер массива: M + 1, так как порядок M означает M+1 коэффициентов
        H = np.zeros(M + 1)
        
        # Цикл расчета коэффициентов фильтра
        # Соответствует строкам 250-290 в BASIC: FOR I% = 0 TO 100
        for i in range(M + 1):
            # Расчет идеальной импульсной характеристики НЧ-фильтра (формула 16.4)
            # Идеальная характеристика: h[i] = sin(2π·fc·(i-M/2)) / (π·(i-M/2))
            
            # Проверка особого случая, когда знаменатель равен нулю
            # Соответствует строке 260 в BASIC: IF (I%-M%/2) = 0 THEN H[I%] = 2*PI*FC
            if (i - M/2) == 0:
                # Особый случай для i = M/2 (избегаем деления на ноль)
                # Используем предел функции: lim(x→0) sin(x)/x = 1
                # Поэтому h[M/2] = 2π·fc
                H[i] = 2 * PI * FC
            else:
                # Фильтр становится физически реализуемым - для вычисления y[j] нужны только x[j], x[j-1], ..., x[j-M]
                # ХЗ почему нельзя брать отсечты больше x[j-M]
                # Основная формула для импульсной характеристики
                # Соответствует строке 270 в BASIC
                H[i] = np.sin(2 * PI * FC * (i - M/2)) / (i - M/2)
            
            # Применение оконной функции для уменьшения эффекта Гиббса
            # Эффект Гиббса - это пульсации в АЧХ, вызванные обрезанием бесконечной импульсной характеристики
            # Соответствует строке 280 в BASIC: H[I%] = H[I%] * (0.54 - 0.46*COS(2*PI*I%/M%) )
            if self.window_type == 'hamming':
                # Окно Хэмминга: w[n] = 0.54 - 0.46·cos(2πn/M)
                # Хороший компромисс между шириной главного лепестка и уровнем боковых лепестков
                H[i] = H[i] * (0.54 - 0.46 * np.cos(2 * PI * i / M))
            elif self.window_type == 'hann':
                # Окно Хэнна: w[n] = 0.5 - 0.5·cos(2πn/M)
                # Более высокие боковые лепестки, но быстрее спадающие
                H[i] = H[i] * (0.5 - 0.5 * np.cos(2 * PI * i / M))
            elif self.window_type == 'blackman':
                # Окно Блэкмана: w[n] = 0.42 - 0.5·cos(2πn/M) + 0.08·cos(4πn/M)
                # Очень низкие боковые лепестки, но шире главный лепесток
                H[i] = H[i] * (0.42 - 0.5 * np.cos(2 * PI * i / M) + 
                               0.08 * np.cos(4 * PI * i / M))
            # Для прямоугольного окна не делаем ничего (коэффициенты остаются без изменения)
            # Прямоугольное окно соответствует исходной программе без оконной функции
        
        # Нормировка коэффициентов для единичного коэффициента усиления на нулевой частоте
        # Это гарантирует, что постоянная составляющая сигнала не изменяется
        # Соответствует строкам 310-380 в BASIC
        # Сначала вычисляется сумма всех коэффициентов
        # Затем каждый коэффициент делится на эту сумму
        H = H / np.sum(H)
        
        # Сохраняем коэффициенты как атрибут объекта для использования в других методах
        self.coefficients = H
        return H
    
    def apply_filter(self, input_signal):
        """
        Применение фильтра к входному сигналу методом свертки
        Соответствует строкам 400-450 оригинальной BASIC-программы
        
        Принцип работы:
        Каждый выходной отсчет вычисляется как взвешенная сумма предыдущих входных отсчетов
        Весами служат коэффициенты фильтра
        
        Параметры:
        ----------
        input_signal : numpy.array
            Входной сигнал для фильтрации
            
        Возвращает:
        -----------
        output_signal : numpy.array
            Отфильтрованный сигнал
        """
        M = self.filter_order      # Порядок фильтра
        H = self.coefficients      # Коэффициенты фильтра
        N = len(input_signal)      # Длина входного сигнала
        
        # Инициализация выходного сигнала нулями (аналог DIM Y[5000] в BASIC)
        output_signal = np.zeros(N)
        
        # Операция свертки - соответствует строкам 400-450 в BASIC
        # В BASIC: FOR J% = 100 TO 4999 (от M до N-1)
        # Проходим по всем отсчетам выходного сигнала, начиная с M-го
        # Это необходимо потому, что для расчета каждого выходного отсчета нужны предыдущие M отсчетов
        for j in range(M, N):
            y_j = 0  # Временная переменная для накопления суммы
            
            # В BASIC: FOR I% = 0 TO 100 (от 0 до M)
            # Внутренний цикл - вычисление свертки для одного отсчета
            for i in range(M + 1):
                # Свертка: y[j] = Σ (x[j-i] · h[i]) для i = 0...M
                # Каждый выходной отсчет - это сумма произведений входных отсчетов на коэффициенты фильтра
                y_j += input_signal[j - i] * H[i]
            
            # Сохраняем вычисленное значение в выходной сигнал
            output_signal[j] = y_j
        
        return output_signal
    
    def get_frequency_response(self, n_points=1024):
        """
        Расчет амплитудно-частотной характеристики (АЧХ) фильтра
        АЧХ показывает, как фильтр воздействует на разные частоты (коэффициент передачи)
        
        Параметры:
        ----------
        n_points : int, optional
            Количество точек для расчета АЧХ, по умолчанию 1024
            Больше точек = более гладкая характеристика
            
        Возвращает:
        -----------
        frequencies : numpy.array
            Массив частот в Герцах
        magnitude : numpy.array
            Амплитуда коэффициента передачи на каждой частоте
        """
        # Используем функцию freqz из scipy.signal для расчета частотной характеристики
        # freqz возвращает комплексную частотную характеристику
        w, h = signal.freqz(self.coefficients, worN=n_points)
        # w - нормированные частоты в радианах/отсчет (от 0 до π)
        # h - комплексная частотная характеристика
        
        # Преобразование нормированных частот в абсолютные в Герцах
        # Формула: f = w * f_sampling / (2π)
        frequencies = w * self.sampling_rate / (2 * np.pi)
        
        # Вычисление амплитуды (модуля) комплексного коэффициента передачи
        magnitude = np.abs(h)
        
        return frequencies, magnitude
    
    def get_window_function(self):
        """
        Получение оконной функции для визуализации
        
        Возвращает:
        -----------
        window : numpy.array
            Массив значений оконной функции
        """
        M = self.filter_order  # Порядок фильтра
        
        # Расчет различных типов оконных функций
        if self.window_type == 'hamming':
            # Окно Хэмминга: w[n] = 0.54 - 0.46·cos(2πn/M)
            window = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(M + 1) / M)
        elif self.window_type == 'hann':
            # Окно Хэнна: w[n] = 0.5 - 0.5·cos(2πn/M)
            window = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(M + 1) / M)
        elif self.window_type == 'blackman':
            # Окно Блэкмана: w[n] = 0.42 - 0.5·cos(2πn/M) + 0.08·cos(4πn/M)
            window = (0.42 - 0.5 * np.cos(2 * np.pi * np.arange(M + 1) / M) + 
                     0.08 * np.cos(4 * np.pi * np.arange(M + 1) / M))
        else:  # rectangular (прямоугольное окно)
            # Прямоугольное окно: все коэффициенты равны 1
            window = np.ones(M + 1)
        
        return window


def generate_test_signal(length, sampling_rate, frequencies, amplitudes=None, noise_level=0.1):
    """
    Генерация тестового сигнала для демонстрации работы фильтра
    
    Создает сигнал как сумму синусоид заданных частот с добавлением шума
    Это позволяет проверить, как фильтр подавляет высокочастотные компоненты
    
    Параметры:
    ----------
    length : int
        Количество отсчетов сигнала (длина сигнала)
    sampling_rate : int
        Частота дискретизации в Герцах
    frequencies : list of float
        Список частот синусоидальных компонент в Герцах
    amplitudes : list of float, optional
        Список амплитуд компонент, по умолчанию единичные амплитуды
    noise_level : float, optional
        Уровень случайного шума (стандартное отклонение), по умолчанию 0.1
        
    Возвращает:
    -----------
    signal_noisy : numpy.array
        Зашумленный сигнал (для фильтрации)
    signal_clean : numpy.array
        Чистый сигнал без шума (для сравнения)
    t : numpy.array
        Временная ось в секундах
    """
    # Если амплитуды не заданы, используем единичные амплитуды для всех частот
    if amplitudes is None:
        amplitudes = [1.0] * len(frequencies)
    
    # Создание временной оси
    # t = [0, 1/f_s, 2/f_s, ..., (length-1)/f_s]
    t = np.arange(length) / sampling_rate
    
    # Инициализация чистого сигнала нулями
    signal_clean = np.zeros(length)
    
    # Создание многочастотного сигнала как суммы синусоид
    # Каждая синусоида: A·sin(2π·f·t)
    for freq, amp in zip(frequencies, amplitudes):
        signal_clean += amp * np.sin(2 * np.pi * freq * t)
    
    # Добавление белого гауссовского шума для реалистичности
    # Шум моделирует реальные условия, когда сигнал содержит случайные помехи
    noise = noise_level * np.random.randn(length)
    signal_noisy = signal_clean + noise
    
    return signal_noisy, signal_clean, t


def compute_spectrum(signal, sampling_rate):
    """
    Вычисление амплитудного спектра сигнала с помощью Быстрого Преобразования Фурье (БПФ)
    
    Принцип работы:
    БПФ разлагает сигнал на синусоидальные компоненты разной частоты
    и показывает их относительные амплитуды
    
    Параметры:
    ----------
    signal : numpy.array
        Входной сигнал для анализа
    sampling_rate : int
        Частота дискретизации сигнала
        
    Возвращает:
    -----------
    frequencies : numpy.array
        Массив частот в Герцах (положительные частоты)
    magnitude : numpy.array
        Амплитудный спектр (нормированные амплитуды)
    """
    N = len(signal)  # Длина сигнала
    
    # Вычисление быстрого преобразования Фурье
    # Возвращает комплексный спектр
    fft_result = np.fft.fft(signal)
    
    # Получение частотной оси
    # np.fft.fftfreq генерирует массив частот для БПФ
    frequencies = np.fft.fftfreq(N, 1/sampling_rate)
    
    # Берем только положительные частоты (односторонний спектр)
    # Так как спектр симметричен относительно нуля для вещественных сигналов
    positive_freq_idx = frequencies >= 0
    frequencies = frequencies[positive_freq_idx]
    
    # Нормировка амплитуды
    # Делим на N для получения правильной амплитуды
    # Умножаем на 2 для компенсации потерь из-за использования только положительных частот
    magnitude = np.abs(fft_result[positive_freq_idx]) / N * 2
    magnitude[0] /= 2  # Постоянная составляющая (DC) не удваивается
    
    return frequencies, magnitude

    ####################################################################################################
    # Графики
    ####################################################################################################
def create_filter_plots(original_signal, filtered_signal, t, sampling_rate, filter_obj):
    """
    Создание и отображение графиков для комплексного анализа работы фильтра
    
    Создает 4 графика, которые показывают:
    1. Сигналы во временной области (до и после фильтрации)
    2. Амплитудные спектры сигналов (частотный анализ)
    3. Оконную функцию фильтра (визуализация окна)
    4. АЧХ фильтра (теоретическая характеристика)
    
    Параметры:
    ----------
    original_signal : numpy.array
        Исходный сигнал до фильтрации
    filtered_signal : numpy.array
        Отфильтрованный сигнал
    t : numpy.array
        Временная ось
    sampling_rate : int
        Частота дискретизации
    filter_obj : WindowedLowPassFilter
        Объект фильтра для получения дополнительных данных
    """
    
    # Вычисление спектров исходного и отфильтрованного сигналов
    freq_orig, mag_orig = compute_spectrum(original_signal, sampling_rate)
    freq_filt, mag_filt = compute_spectrum(filtered_signal, sampling_rate)
    
    # Получение АЧХ фильтра (теоретической частотной характеристики)
    freq_response, mag_response = filter_obj.get_frequency_response()
    
    # Получение оконной функции для визуализации
    window = filter_obj.get_window_function()
    window_x = np.arange(len(window))  # Ось для оконной функции
    
    # 1. График сигналов во временной области
    # Показывает, как фильтр изменяет сигнал во времени
    p1 = figure(
        title="Сигналы во временной области", 
        width=600, height=300,
        x_axis_label='Время (с)', 
        y_axis_label='Амплитуда'
    )
    # Исходный сигнал (синяя линия)
    p1.line(t, original_signal, 
            legend_label='Исходный сигнал', 
            line_color='blue', line_width=2)
    # Отфильтрованный сигнал (красная линия)
    p1.line(t, filtered_signal, 
            legend_label='Отфильтрованный сигнал', 
            line_color='red', line_width=2)
    p1.legend.location = "top_right"  # Размещение легенды
    
    # 2. График амплитудных спектров
    # Показывает, какие частоты присутствуют в сигналах
    p2 = figure(
        title="Спектры сигналов", 
        width=600, height=300,
        x_axis_label='Частота (Гц)', 
        y_axis_label='Амплитуда'
    )
    # Спектр исходного сигнала
    p2.line(freq_orig, mag_orig, 
            legend_label='Исходный спектр', 
            line_color='blue', line_width=2)
    # Спектр отфильтрованного сигнала
    p2.line(freq_filt, mag_filt, 
            legend_label='Отфильтрованный спектр', 
            line_color='red', line_width=2)
    p2.legend.location = "top_right"
    
    # 3. График оконной функции
    # Показывает форму окна, применяемого к коэффициентам фильтра
    p3 = figure(
        title="Оконная функция", 
        width=600, height=300,
        x_axis_label='Отсчеты', 
        y_axis_label='Амплитуда'
    )
    p3.line(window_x, window, 
            line_color='green', line_width=2)
    p3.circle(window_x, window, 
              size=5, color='green', alpha=0.7)  # Точки для наглядности
    
    # 4. График АЧХ фильтра
    # Показывает теоретическую частотную характеристику фильтра
    p4 = figure(
        title="АЧХ фильтра", 
        width=600, height=300,
        x_axis_label='Частота (Гц)', 
        y_axis_label='Коэффициент передачи'
    )
    # Основная АЧХ фильтра
    p4.line(freq_response, mag_response, 
            line_color='purple', line_width=2)
    
    # Вертикальная линия для обозначения частоты среза
    # Показывает, где находится граница полосы пропускания
    p4.line([filter_obj.cutoff_freq, filter_obj.cutoff_freq], 
            [0, max(mag_response)], 
            line_color='red', 
            line_dash='dashed',  # Пунктирная линия
            line_width=1, 
            legend_label=f'f_cut = {filter_obj.cutoff_freq} Гц')
    p4.legend.location = "top_right"
    
    # Создание сетки 2x2 и отображение всех графиков
    # gridplot размещает графики в таблице 2 строки × 2 столбца
    plot_grid = gridplot([[p1, p2], [p3, p4]])
    show(plot_grid)  # Отображение всей сетки графиков


def demo_lowpass_filter(sampling_rate=1000, cutoff_freq=100, filter_order=100, 
                       window_type='hamming', signal_length=2000,
                       frequencies=[50, 150, 300], amplitudes=[1.0, 0.5, 0.3]):
    """
    Основная функция демонстрации работы фильтра нижних частот
    
    Эта функция объединяет все этапы:
    1. Генерация тестового сигнала
    2. Создание и настройка фильтра
    3. Применение фильтра
    4. Визуализация результатов
    5. Вывод информации о параметрах
    
    Параметры:
    ----------
    sampling_rate : int, optional
        Частота дискретизации в Герцах, по умолчанию 1000 Гц
    cutoff_freq : float, optional
        Частота среза в Герцах, по умолчанию 100 Гц
    filter_order : int, optional
        Порядок фильтра, по умолчанию 100
    window_type : str, optional
        Тип окна, по умолчанию 'hamming'
    signal_length : int, optional
        Длина тестового сигнала в отсчетах, по умолчанию 2000
    frequencies : list of float, optional
        Частоты компонент тестового сигнала, по умолчанию [50, 150, 300] Гц
    amplitudes : list of float, optional
        Амплитуды компонент тестового сигнала, по умолчанию [1.0, 0.5, 0.3]
        
    Возвращает:
    -----------
    test_signal : numpy.array
        Исходный тестовый сигнал
    filtered_signal : numpy.array
        Отфильтрованный сигнал
    lp_filter : WindowedLowPassFilter
        Объект фильтра для дальнейшего анализа
    """
    
    # Вывод заголовка демонстрации
    print("=" * 60)
    print("ФИЛЬТР НИЖНИХ ЧАСТОТ С ОКОННЫМИ ФУНКЦИЯМИ")
    print("=" * 60)
    
    # 1. Генерация тестового сигнала (аналог строки 220 в BASIC: GOSUB XXXX)
    test_signal, clean_signal, time = generate_test_signal(
        signal_length, sampling_rate, frequencies, amplitudes, noise_level=0.2
    )
    
    # 2. Создание и проектирование фильтра
    lp_filter = WindowedLowPassFilter(sampling_rate, cutoff_freq, filter_order, window_type)
    coefficients = lp_filter.design_filter()  # Расчет коэффициентов
    
    # 3. Применение фильтра к сигналу
    filtered_signal = lp_filter.apply_filter(test_signal)
    
    # 4. Создание и отображение графиков для визуального анализа
    create_filter_plots(test_signal, filtered_signal, time, sampling_rate, lp_filter)
    
    # 5. Вывод подробной информации о параметрах для изучения
    print(f"ПАРАМЕТРЫ ФИЛЬТРА:")
    print(f"  Частота дискретизации: {sampling_rate} Гц")
    print(f"  Частота среза: {cutoff_freq} Гц")
    print(f"  Порядок фильтра: {filter_order}")
    print(f"  Тип окна: {window_type}")
    print(f"  Нормированная частота среза: {lp_filter.FC:.4f}")
    print(f"  Сумма коэффициентов фильтра: {np.sum(coefficients):.6f} (должна быть ≈ 1.0)")
    print()
    print(f"ПАРАМЕТРЫ СИГНАЛА:")
    print(f"  Длина сигнала: {signal_length} отсчетов")
    print(f"  Длительность сигнала: {signal_length/sampling_rate:.2f} секунд")
    print(f"  Компоненты: {[f'{f} Гц (A={a})' for f, a in zip(frequencies, amplitudes)]}")
    print(f"  Частота Найквиста: {sampling_rate/2} Гц")
    print()
    
    return test_signal, filtered_signal, lp_filter


# Запуск демонстрации работы фильтра
if __name__ == "__main__":
    """
    Этот блок выполнится только при прямом запуске скрипта как программы
    В Jupyter Notebook можно вызывать функцию demo_lowpass_filter() напрямую в ячейках
    """
    print("Запуск демонстрации фильтра нижних частот...")
    test_signal, filtered_signal, filter_obj = demo_lowpass_filter()

Loading BokehJS ...

Запуск демонстрации фильтра нижних частот...
ФИЛЬТР НИЖНИХ ЧАСТОТ С ОКОННЫМИ ФУНКЦИЯМИ


ПАРАМЕТРЫ ФИЛЬТРА:
  Частота дискретизации: 1000 Гц
  Частота среза: 100 Гц
  Порядок фильтра: 100
  Тип окна: hamming
  Нормированная частота среза: 0.2000
  Сумма коэффициентов фильтра: 1.000000 (должна быть ≈ 1.0)

ПАРАМЕТРЫ СИГНАЛА:
  Длина сигнала: 2000 отсчетов
  Длительность сигнала: 2.00 секунд
  Компоненты: ['50 Гц (A=1.0)', '150 Гц (A=0.5)', '300 Гц (A=0.3)']
  Частота Найквиста: 500.0 Гц



In [7]:
# Импорт необходимых библиотек
# numpy - для математических операций и работы с массивами
import numpy as np
# Bokeh - для создания интерактивных графиков
from bokeh.plotting import figure, show  # figure - создание графиков, show - отображение
from bokeh.layouts import gridplot, column  # gridplot - компоновка графиков в сетку
from bokeh.io import output_notebook     # output_notebook - вывод в Jupyter Notebook
# scipy.signal - для сигнальной обработки (расчет АЧХ)
from scipy import signal

# Активируем вывод графиков Bokeh непосредственно в Jupyter Notebook
# Это позволяет отображать графики прямо в ячейках ноутбука
output_notebook()

class WindowedLowPassFilter:
    """
    Класс для создания и применения оконного КИХ-фильтра нижних частот
    (КИХ = Конечная Импульсная Характеристика)
    
    Основные принципы:
    - Фильтр работает методом свертки входного сигнала с импульсной характеристикой
    - Используется метод окон для уменьшения эффекта Гиббса (пульсации в АЧХ)
    - Сохраняет логику оригинальной BASIC-программы 16.1
    - Дополнительно: свертка импульсной характеристики самой с собой для улучшения характеристик
    
    Математическая основа:
    Импульсная характеристика идеального НЧ-фильтра: h[n] = sin(2π·fc·n) / (π·n)
    где fc - нормированная частота среза
    """
    
    def __init__(self, sampling_rate, cutoff_freq, filter_order, window_type='hamming', self_convolution=False):
        """
        Конструктор класса - инициализирует основные параметры фильтра
        
        Параметры:
        ----------
        sampling_rate : int
            Частота дискретизации в Герцах (количество отсчетов в секунду)
            Определяет максимальную частоту по теореме Найквиста (f_nyquist = sampling_rate/2)
            
        cutoff_freq : float
            Граничная частота среза в Герцах. Частоты выше этой будут ослабляться фильтром
            
        filter_order : int
            Порядок фильтра (количество коэффициентов минус 1)
            Определяет крутизну склона АЧХ и вычислительную сложность
            Чем выше порядок - тем резче переходная полоса, но больше задержка
            
        window_type : str, optional
            Тип оконной функции для уменьшения эффекта Гиббса, по умолчанию 'hamming'
            Доступные варианты: 'hamming', 'hann', 'blackman', 'rectangular'
            Окна отличаются компромиссом между шириной главного лепестка и уровнем боковых лепестков
            
        self_convolution : bool, optional
            Флаг применения свертки импульсной характеристики самой с собой
            Если True, то после расчета исходной ИХ выполняется ее свертка сама с собой
            Это улучшает крутизну склона АЧХ, но увеличивает порядок фильтра
        """
        # Сохраняем параметры фильтра как атрибуты объекта
        self.sampling_rate = sampling_rate  # Частота дискретизации в Гц
        self.cutoff_freq = cutoff_freq      # Частота среза в Гц
        self.filter_order = filter_order    # Порядок фильтра
        self.window_type = window_type      # Тип оконной функции
        self.self_convolution = self_convolution  # Флаг самосвертки
        self.PI = np.pi                     # Константа π для математических расчетов
        
        # Нормированная частота среза (относительно частоты Найквиста)
        # В цифровой обработке частоты обычно нормируются к половине частоты дискретизации
        # В оригинальной BASIC-программе FC задавалась в диапазоне 0...0.5
        # Формула: FC = f_cutoff / (f_sampling / 2) = 2 * f_cutoff / f_sampling
        self.FC = cutoff_freq / (sampling_rate / 2)
        
    def design_filter(self):
        """
        Расчет весовых коэффициентов фильтра (импульсной характеристики)
        Соответствует строкам 240-290 оригинальной BASIC-программы
        
        Математическая основа:
        - Сначала рассчитывается идеальная импульсная характеристика НЧ-фильтра
        - Затем применяется оконная функция для уменьшения эффекта Гиббса
        - Наконец, коэффициенты нормируются для единичного коэффициента усиления
        - ДОПОЛНИТЕЛЬНО: если включена самосвертка, ИХ свертывается сама с собой
        
        Возвращает:
        -----------
        H : numpy.array
            Массив коэффициентов фильтра (импульсная характеристика)
        """
        # Локальные переменные для удобства чтения кода
        M = self.filter_order  # Порядок фильтра
        FC = self.FC           # Нормированная частота среза
        PI = self.PI           # Число π
        
        # Инициализация массива коэффициентов (аналог DIM H[100] в BASIC)
        # Размер массива: M + 1, так как порядок M означает M+1 коэффициентов
        H = np.zeros(M + 1)
        
        # Цикл расчета коэффициентов фильтра
        # Соответствует строкам 250-290 в BASIC: FOR I% = 0 TO 100
        for i in range(M + 1):
            # Расчет идеальной импульсной характеристики НЧ-фильтра (формула 16.4)
            # Идеальная характеристика: h[i] = sin(2π·fc·(i-M/2)) / (π·(i-M/2))
            
            # Проверка особого случая, когда знаменатель равен нулю
            # Соответствует строке 260 в BASIC: IF (I%-M%/2) = 0 THEN H[I%] = 2*PI*FC
            if (i - M/2) == 0:
                # Особый случай для i = M/2 (избегаем деления на ноль)
                # Используем предел функции: lim(x→0) sin(x)/x = 1
                # Поэтому h[M/2] = 2π·fc
                H[i] = 2 * PI * FC
            else:
                # Основная формула для импульсной характеристики
                # Соответствует строке 270 в BASIC
                H[i] = np.sin(2 * PI * FC * (i - M/2)) / (i - M/2)
            
            # Применение оконной функции для уменьшения эффекта Гиббса
            # Эффект Гиббса - это пульсации в АЧХ, вызванные обрезанием бесконечной импульсной характеристики
            # Соответствует строке 280 в BASIC: H[I%] = H[I%] * (0.54 - 0.46*COS(2*PI*I%/M%) )
            if self.window_type == 'hamming':
                # Окно Хэмминга: w[n] = 0.54 - 0.46·cos(2πn/M)
                # Хороший компромисс между шириной главного лепестка и уровнем боковых лепестков
                H[i] = H[i] * (0.54 - 0.46 * np.cos(2 * PI * i / M))
            elif self.window_type == 'hann':
                # Окно Хэнна: w[n] = 0.5 - 0.5·cos(2πn/M)
                # Более высокие боковые лепестки, но быстрее спадающие
                H[i] = H[i] * (0.5 - 0.5 * np.cos(2 * PI * i / M))
            elif self.window_type == 'blackman':
                # Окно Блэкмана: w[n] = 0.42 - 0.5·cos(2πn/M) + 0.08·cos(4πn/M)
                # Очень низкие боковые лепестки, но шире главный лепесток
                H[i] = H[i] * (0.42 - 0.5 * np.cos(2 * PI * i / M) + 
                               0.08 * np.cos(4 * PI * i / M))
            # Для прямоугольного окна не делаем ничего (коэффициенты остаются без изменения)
            # Прямоугольное окно соответствует исходной программе без оконной функции
        
        # Нормировка коэффициентов для единичного коэффициента усиления на нулевой частоте
        # Это гарантирует, что постоянная составляющая сигнала не изменяется
        # Соответствует строкам 310-380 в BASIC
        # Сначала вычисляется сумма всех коэффициентов
        # Затем каждый коэффициент делится на эту сумму
        H = H / np.sum(H)
        
        # Сохраняем исходную импульсную характеристику для анализа и сравнения
        self.coefficients_original = H.copy()
        
        # ДОПОЛНИТЕЛЬНАЯ ФУНКЦИОНАЛЬНОСТЬ: Свертка импульсной характеристики самой с собой
        # Это улучшает характеристики фильтра, но увеличивает его порядок
        if self.self_convolution:
            print("Применение свертки импульсной характеристики самой с собой...")
            
            # Свертка исходной ИХ самой с собой
            # Математически: h_conv[n] = (h_original * h_original)[n]
            # Это эквивалентно последовательному применению двух одинаковых фильтров
            H_conv = np.convolve(H, H, mode='full')
            
            # Нормировка свернутой импульсной характеристики
            H_conv = H_conv / np.sum(H_conv)
            
            # Обновляем коэффициенты фильтра
            H = H_conv
            
            # Обновляем порядок фильтра (после свертки порядок увеличивается)
            self.effective_order = len(H) - 1
            print(f"Порядок фильтра увеличен с {self.filter_order} до {self.effective_order}")
        else:
            self.effective_order = self.filter_order
        
        # Сохраняем финальные коэффициенты как атрибут объекта для использования в других методах
        self.coefficients = H
        return H
    
    def apply_filter(self, input_signal):
        """
        Применение фильтра к входному сигналу методом свертки
        Соответствует строкам 400-450 оригинальной BASIC-программы
        
        Принцип работы:
        Каждый выходной отсчет вычисляется как взвешенная сумма предыдущих входных отсчетов
        Весами служат коэффициенты фильтра
        
        Параметры:
        ----------
        input_signal : numpy.array
            Входной сигнал для фильтрации
            
        Возвращает:
        -----------
        output_signal : numpy.array
            Отфильтрованный сигнал
        """
        # Используем эффективный порядок фильтра (учитывает самосвертку)
        M = self.effective_order
        H = self.coefficients      # Коэффициенты фильтра
        N = len(input_signal)      # Длина входного сигнала
        
        # Инициализация выходного сигнала нулями (аналог DIM Y[5000] в BASIC)
        output_signal = np.zeros(N)
        
        # Операция свертки - соответствует строкам 400-450 в BASIC
        # В BASIC: FOR J% = 100 TO 4999 (от M до N-1)
        # Проходим по всем отсчетам выходного сигнала, начиная с M-го
        # Это необходимо потому, что для расчета каждого выходного отсчета нужны предыдущие M отсчетов
        for j in range(M, N):
            y_j = 0  # Временная переменная для накопления суммы
            
            # В BASIC: FOR I% = 0 TO 100 (от 0 до M)
            # Внутренний цикл - вычисление свертки для одного отсчета
            for i in range(M + 1):
                # Свертка: y[j] = Σ (x[j-i] · h[i]) для i = 0...M
                # Каждый выходной отсчет - это сумма произведений входных отсчетов на коэффициенты фильтра
                y_j += input_signal[j - i] * H[i]
            
            # Сохраняем вычисленное значение в выходной сигнал
            output_signal[j] = y_j
        
        return output_signal
    
    def get_frequency_response(self, n_points=1024):
        """
        Расчет амплитудно-частотной характеристики (АЧХ) фильтра
        АЧХ показывает, как фильтр воздействует на разные частоты (коэффициент передачи)
        
        Параметры:
        ----------
        n_points : int, optional
            Количество точек для расчета АЧХ, по умолчанию 1024
            Больше точек = более гладкая характеристика
            
        Возвращает:
        -----------
        frequencies : numpy.array
            Массив частот в Герцах
        magnitude : numpy.array
            Амплитуда коэффициента передачи на каждой частоте
        """
        # Используем функцию freqz из scipy.signal для расчета частотной характеристики
        # freqz возвращает комплексную частотную характеристику
        w, h = signal.freqz(self.coefficients, worN=n_points)
        # w - нормированные частоты в радианах/отсчет (от 0 до π)
        # h - комплексная частотная характеристика
        
        # Преобразование нормированных частот в абсолютные в Герцах
        # Формула: f = w * f_sampling / (2π)
        frequencies = w * self.sampling_rate / (2 * np.pi)
        
        # Вычисление амплитуды (модуля) комплексного коэффициента передачи
        magnitude = np.abs(h)
        
        return frequencies, magnitude
    
    def get_original_frequency_response(self, n_points=1024):
        """
        Расчет АЧХ исходного фильтра (до самосвертки)
        Полезно для сравнения характеристик до и после улучшения
        
        Параметры:
        ----------
        n_points : int, optional
            Количество точек для расчета АЧХ, по умолчанию 1024
            
        Возвращает:
        -----------
        frequencies : numpy.array
            Массив частот в Герцах
        magnitude : numpy.array
            Амплитуда коэффициента передачи на каждой частоте
        """
        # Проверяем, есть ли исходные коэффициенты для сравнения
        if hasattr(self, 'coefficients_original'):
            w, h = signal.freqz(self.coefficients_original, worN=n_points)
            frequencies = w * self.sampling_rate / (2 * np.pi)
            magnitude = np.abs(h)
            return frequencies, magnitude
        else:
            # Если исходных коэффициентов нет, возвращаем текущую АЧХ
            return self.get_frequency_response(n_points)
    
    def get_window_function(self):
        """
        Получение оконной функции для визуализации
        
        Возвращает:
        -----------
        window : numpy.array
            Массив значений оконной функции
        """
        # Используем исходный порядок фильтра для оконной функции
        M = self.filter_order
        
        # Расчет различных типов оконных функций
        if self.window_type == 'hamming':
            # Окно Хэмминга: w[n] = 0.54 - 0.46·cos(2πn/M)
            window = 0.54 - 0.46 * np.cos(2 * np.pi * np.arange(M + 1) / M)
        elif self.window_type == 'hann':
            # Окно Хэнна: w[n] = 0.5 - 0.5·cos(2πn/M)
            window = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(M + 1) / M)
        elif self.window_type == 'blackman':
            # Окно Блэкмана: w[n] = 0.42 - 0.5·cos(2πn/M) + 0.08·cos(4πn/M)
            window = (0.42 - 0.5 * np.cos(2 * np.pi * np.arange(M + 1) / M) + 
                     0.08 * np.cos(4 * np.pi * np.arange(M + 1) / M))
        else:  # rectangular (прямоугольное окно)
            # Прямоугольное окно: все коэффициенты равны 1
            window = np.ones(M + 1)
        
        return window


def generate_test_signal(length, sampling_rate, frequencies, amplitudes=None, noise_level=0.1):
    """
    Генерация тестового сигнала для демонстрации работы фильтра
    
    Создает сигнал как сумму синусоид заданных частот с добавлением шума
    Это позволяет проверить, как фильтр подавляет высокочастотные компоненты
    
    Параметры:
    ----------
    length : int
        Количество отсчетов сигнала (длина сигнала)
    sampling_rate : int
        Частота дискретизации в Герцах
    frequencies : list of float
        Список частот синусоидальных компонент в Герцах
    amplitudes : list of float, optional
        Список амплитуд компонент, по умолчанию единичные амплитуды
    noise_level : float, optional
        Уровень случайного шума (стандартное отклонение), по умолчанию 0.1
        
    Возвращает:
    -----------
    signal_noisy : numpy.array
        Зашумленный сигнал (для фильтрации)
    signal_clean : numpy.array
        Чистый сигнал без шума (для сравнения)
    t : numpy.array
        Временная ось в секундах
    """
    # Если амплитуды не заданы, используем единичные амплитуды для всех частот
    if amplitudes is None:
        amplitudes = [1.0] * len(frequencies)
    
    # Создание временной оси
    # t = [0, 1/f_s, 2/f_s, ..., (length-1)/f_s]
    t = np.arange(length) / sampling_rate
    
    # Инициализация чистого сигнала нулями
    signal_clean = np.zeros(length)
    
    # Создание многочастотного сигнала как суммы синусоид
    # Каждая синусоида: A·sin(2π·f·t)
    for freq, amp in zip(frequencies, amplitudes):
        signal_clean += amp * np.sin(2 * np.pi * freq * t)
    
    # Добавление белого гауссовского шума для реалистичности
    # Шум моделирует реальные условия, когда сигнал содержит случайные помехи
    noise = noise_level * np.random.randn(length)
    signal_noisy = signal_clean + noise
    
    return signal_noisy, signal_clean, t


def compute_spectrum(signal, sampling_rate):
    """
    Вычисление амплитудного спектра сигнала с помощью Быстрого Преобразования Фурье (БПФ)
    
    Принцип работы:
    БПФ разлагает сигнал на синусоидальные компоненты разной частоты
    и показывает их относительные амплитуды
    
    Параметры:
    ----------
    signal : numpy.array
        Входной сигнал для анализа
    sampling_rate : int
        Частота дискретизации сигнала
        
    Возвращает:
    -----------
    frequencies : numpy.array
        Массив частот в Герцах (положительные частоты)
    magnitude : numpy.array
        Амплитудный спектр (нормированные амплитуды)
    """
    N = len(signal)  # Длина сигнала
    
    # Вычисление быстрого преобразования Фурье
    # Возвращает комплексный спектр
    fft_result = np.fft.fft(signal)
    
    # Получение частотной оси
    # np.fft.fftfreq генерирует массив частот для БПФ
    frequencies = np.fft.fftfreq(N, 1/sampling_rate)
    
    # Берем только положительные частоты (односторонний спектр)
    # Так как спектр симметричен относительно нуля для вещественных сигналов
    positive_freq_idx = frequencies >= 0
    frequencies = frequencies[positive_freq_idx]
    
    # Нормировка амплитуды
    # Делим на N для получения правильной амплитуды
    # Умножаем на 2 для компенсации потерь из-за использования только положительных частот
    magnitude = np.abs(fft_result[positive_freq_idx]) / N * 2
    magnitude[0] /= 2  # Постоянная составляющая (DC) не удваивается
    
    return frequencies, magnitude

    ####################################################################################################
    # Графики
    ####################################################################################################
def create_filter_plots(original_signal, filtered_signal, t, sampling_rate, filter_obj):
    """
    Создание и отображение графиков для комплексного анализа работы фильтра
    
    Создает 5 графиков, которые показывают:
    1. Сигналы во временной области (до и после фильтрации)
    2. Амплитудные спектры сигналов (частотный анализ)
    3. Оконную функцию фильтра (визуализация окна)
    4. АЧХ фильтра (теоретическая характеристика)
    5. Импульсные характеристики (до и после самосвертки)
    
    Параметры:
    ----------
    original_signal : numpy.array
        Исходный сигнал до фильтрации
    filtered_signal : numpy.array
        Отфильтрованный сигнал
    t : numpy.array
        Временная ось
    sampling_rate : int
        Частота дискретизации
    filter_obj : WindowedLowPassFilter
        Объект фильтра для получения дополнительных данных
    """
    
    # Вычисление спектров исходного и отфильтрованного сигналов
    freq_orig, mag_orig = compute_spectrum(original_signal, sampling_rate)
    freq_filt, mag_filt = compute_spectrum(filtered_signal, sampling_rate)
    
    # Получение АЧХ фильтра (теоретической частотной характеристики)
    freq_response, mag_response = filter_obj.get_frequency_response()
    
    # Получение оконной функции для визуализации
    window = filter_obj.get_window_function()
    window_x = np.arange(len(window))  # Ось для оконной функции
    
    # 1. График сигналов во временной области
    # Показывает, как фильтр изменяет сигнал во времени
    p1 = figure(
        title="Сигналы во временной области", 
        width=600, height=300,
        x_axis_label='Время (с)', 
        y_axis_label='Амплитуда'
    )
    # Исходный сигнал (синяя линия)
    p1.line(t, original_signal, 
            legend_label='Исходный сигнал', 
            line_color='blue', line_width=2)
    # Отфильтрованный сигнал (красная линия)
    p1.line(t, filtered_signal, 
            legend_label='Отфильтрованный сигнал', 
            line_color='red', line_width=2)
    p1.legend.location = "top_right"  # Размещение легенды
    
    # 2. График амплитудных спектров
    # Показывает, какие частоты присутствуют в сигналах
    p2 = figure(
        title="Спектры сигналов", 
        width=600, height=300,
        x_axis_label='Частота (Гц)', 
        y_axis_label='Амплитуда'
    )
    # Спектр исходного сигнала
    p2.line(freq_orig, mag_orig, 
            legend_label='Исходный спектр', 
            line_color='blue', line_width=2)
    # Спектр отфильтрованного сигнала
    p2.line(freq_filt, mag_filt, 
            legend_label='Отфильтрованный спектр', 
            line_color='red', line_width=2)
    p2.legend.location = "top_right"
    
    # 3. График оконной функции
    # Показывает форму окна, применяемого к коэффициентам фильтра
    p3 = figure(
        title="Оконная функция", 
        width=600, height=300,
        x_axis_label='Отсчеты', 
        y_axis_label='Амплитуда'
    )
    p3.line(window_x, window, 
            line_color='green', line_width=2)
    p3.circle(window_x, window, 
              size=5, color='green', alpha=0.7)  # Точки для наглядности
    
    # 4. График АЧХ фильтра
    # Показывает теоретическую частотную характеристику фильтра
    p4 = figure(
        title="АЧХ фильтра", 
        width=600, height=300,
        x_axis_label='Частота (Гц)', 
        y_axis_label='Коэффициент передачи'
    )
    
    # Если доступны исходные коэффициенты (была применена самосвертка),
    # показываем обе АЧХ для сравнения
    if hasattr(filter_obj, 'coefficients_original') and filter_obj.self_convolution:
        # АЧХ исходного фильтра
        freq_response_orig, mag_response_orig = filter_obj.get_original_frequency_response()
        p4.line(freq_response_orig, mag_response_orig, 
                legend_label='Исходная АЧХ', 
                line_color='blue', line_width=2, line_dash='dashed')
        
        # АЧХ фильтра после самосвертки
        p4.line(freq_response, mag_response, 
                legend_label='АЧХ после самосвертки', 
                line_color='red', line_width=2)
        
        p4.legend.location = "top_right"
    else:
        # Только одна АЧХ
        p4.line(freq_response, mag_response, 
                line_color='purple', line_width=2)
    
    # Вертикальная линия для обозначения частоты среза
    # Показывает, где находится граница полосы пропускания
    p4.line([filter_obj.cutoff_freq, filter_obj.cutoff_freq], 
            [0, max(mag_response)], 
            line_color='red', 
            line_dash='dashed',  # Пунктирная линия
            line_width=1, 
            legend_label=f'f_cut = {filter_obj.cutoff_freq} Гц')
    p4.legend.location = "top_right"
    
    # 5. График импульсных характеристик (НОВЫЙ ГРАФИК)
    # Показывает, как изменилась импульсная характеристика после самосвертки
    p5 = figure(
        title="Импульсные характеристики фильтра", 
        width=600, height=300,
        x_axis_label='Отсчеты', 
        y_axis_label='Амплитуда'
    )
    
    # Если доступны исходные коэффициенты (была применена самосвертка),
    # показываем обе импульсные характеристики для сравнения
    if hasattr(filter_obj, 'coefficients_original') and filter_obj.self_convolution:
        # Исходная импульсная характеристика
        orig_impulse_x = np.arange(len(filter_obj.coefficients_original))
        p5.line(orig_impulse_x, filter_obj.coefficients_original, 
                legend_label='Исходная ИХ', 
                line_color='blue', line_width=2)
        
        # Импульсная характеристика после самосвертки
        conv_impulse_x = np.arange(len(filter_obj.coefficients))
        p5.line(conv_impulse_x, filter_obj.coefficients, 
                legend_label='ИХ после самосвертки', 
                line_color='red', line_width=2)
        
        p5.legend.location = "top_right"
    else:
        # Только одна импульсная характеристика
        impulse_x = np.arange(len(filter_obj.coefficients))
        p5.line(impulse_x, filter_obj.coefficients, 
                line_color='purple', line_width=2)
    
    # Создание сетки 3x2 и отображение всех графиков
    # gridplot размещает графики в таблице 3 строки × 2 столбца
    plot_grid = gridplot([[p1, p2], [p3, p4], [p5, None]])
    show(plot_grid)  # Отображение всей сетки графиков


def demo_lowpass_filter(sampling_rate=500, cutoff_freq=100, filter_order=10000, 
                       window_type='hamming', signal_length=2000,
                       frequencies=[50, 150, 300], amplitudes=[1.0, 0.5, 0.3],
                       self_convolution=False):
    """
    Основная функция демонстрации работы фильтра нижних частот
    
    Эта функция объединяет все этапы:
    1. Генерация тестового сигнала
    2. Создание и настройка фильтра
    3. Применение фильтра
    4. Визуализация результатов
    5. Вывод информации о параметрах
    
    Параметры:
    ----------
    sampling_rate : int, optional
        Частота дискретизации в Герцах, по умолчанию 1000 Гц
    cutoff_freq : float, optional
        Частота среза в Герцах, по умолчанию 100 Гц
    filter_order : int, optional
        Порядок фильтра, по умолчанию 100
    window_type : str, optional
        Тип окна, по умолчанию 'hamming'
    signal_length : int, optional
        Длина тестового сигнала в отсчетах, по умолчанию 2000
    frequencies : list of float, optional
        Частоты компонент тестового сигнала, по умолчанию [50, 150, 300] Гц
    amplitudes : list of float, optional
        Амплитуды компонент тестового сигнала, по умолчанию [1.0, 0.5, 0.3]
    self_convolution : bool, optional
        Применять ли свертку импульсной характеристики самой с собой для улучшения характеристик
        По умолчанию False
        
    Возвращает:
    -----------
    test_signal : numpy.array
        Исходный тестовый сигнал
    filtered_signal : numpy.array
        Отфильтрованный сигнал
    lp_filter : WindowedLowPassFilter
        Объект фильтра для дальнейшего анализа
    """
    
    # Вывод заголовка демонстрации
    print("=" * 60)
    print("ФИЛЬТР НИЖНИХ ЧАСТОТ С ОКОННЫМИ ФУНКЦИЯМИ")
    if self_convolution:
        print("С ПРИМЕНЕНИЕМ САМОСВЕРТКИ ДЛЯ ПОВЫШЕНИЯ ТОЧНОСТИ")
    print("=" * 60)
    
    # 1. Генерация тестового сигнала (аналог строки 220 в BASIC: GOSUB XXXX)
    test_signal, clean_signal, time = generate_test_signal(
        signal_length, sampling_rate, frequencies, amplitudes, noise_level=0.2
    )
    
    # 2. Создание и проектирование фильтра
    lp_filter = WindowedLowPassFilter(sampling_rate, cutoff_freq, filter_order, 
                                     window_type, self_convolution)
    coefficients = lp_filter.design_filter()  # Расчет коэффициентов
    
    # 3. Применение фильтра к сигналу
    filtered_signal = lp_filter.apply_filter(test_signal)
    
    # 4. Создание и отображение графиков для визуального анализа
    create_filter_plots(test_signal, filtered_signal, time, sampling_rate, lp_filter)
    
    # 5. Вывод подробной информации о параметрах для изучения
    print(f"ПАРАМЕТРЫ ФИЛЬТРА:")
    print(f"  Частота дискретизации: {sampling_rate} Гц")
    print(f"  Частота среза: {cutoff_freq} Гц")
    print(f"  Заданный порядок фильтра: {filter_order}")
    if self_convolution:
        print(f"  Эффективный порядок после самосвертки: {lp_filter.effective_order}")
    print(f"  Тип окна: {window_type}")
    print(f"  Нормированная частота среза: {lp_filter.FC:.4f}")
    print(f"  Сумма коэффициентов фильтра: {np.sum(coefficients):.6f} (должна быть ≈ 1.0)")
    if self_convolution:
        print(f"  Самосвертка импульсной характеристики: ДА")
        print(f"  Улучшение: более крутой склон АЧХ, лучшая подавление в полосе задерживания")
    else:
        print(f"  Самосвертка импульсной характеристики: НЕТ")
    print()
    print(f"ПАРАМЕТРЫ СИГНАЛА:")
    print(f"  Длина сигнала: {signal_length} отсчетов")
    print(f"  Длительность сигнала: {signal_length/sampling_rate:.2f} секунд")
    print(f"  Компоненты: {[f'{f} Гц (A={a})' for f, a in zip(frequencies, amplitudes)]}")
    print(f"  Частота Найквиста: {sampling_rate/2} Гц")
    print()
    
    return test_signal, filtered_signal, lp_filter


# Запуск демонстрации работы фильтра
if __name__ == "__main__":
    """
    Этот блок выполнится только при прямом запуске скрипта как программы
    В Jupyter Notebook можно вызывать функцию demo_lowpass_filter() напрямую в ячейках
    """
    print("Запуск демонстрации фильтра нижних частот...")
    
    # Демонстрация без самосвертки
    print("\n1. ДЕМОНСТРАЦИЯ БЕЗ САМОСВЕРТКИ:")
    test_signal1, filtered_signal1, filter_obj1 = demo_lowpass_filter(
        self_convolution=False
    )
    
    # Демонстрация с самосверткой
    print("\n2. ДЕМОНСТРАЦИЯ С САМОСВЕРТКОЙ:")
    test_signal2, filtered_signal2, filter_obj2 = demo_lowpass_filter(
        self_convolution=True
    )

Loading BokehJS ...

Запуск демонстрации фильтра нижних частот...

1. ДЕМОНСТРАЦИЯ БЕЗ САМОСВЕРТКИ:
ФИЛЬТР НИЖНИХ ЧАСТОТ С ОКОННЫМИ ФУНКЦИЯМИ


ПАРАМЕТРЫ ФИЛЬТРА:
  Частота дискретизации: 500 Гц
  Частота среза: 100 Гц
  Заданный порядок фильтра: 10000
  Тип окна: hamming
  Нормированная частота среза: 0.4000
  Сумма коэффициентов фильтра: 1.000000 (должна быть ≈ 1.0)
  Самосвертка импульсной характеристики: НЕТ

ПАРАМЕТРЫ СИГНАЛА:
  Длина сигнала: 2000 отсчетов
  Длительность сигнала: 4.00 секунд
  Компоненты: ['50 Гц (A=1.0)', '150 Гц (A=0.5)', '300 Гц (A=0.3)']
  Частота Найквиста: 250.0 Гц


2. ДЕМОНСТРАЦИЯ С САМОСВЕРТКОЙ:
ФИЛЬТР НИЖНИХ ЧАСТОТ С ОКОННЫМИ ФУНКЦИЯМИ
С ПРИМЕНЕНИЕМ САМОСВЕРТКИ ДЛЯ ПОВЫШЕНИЯ ТОЧНОСТИ
Применение свертки импульсной характеристики самой с собой...
Порядок фильтра увеличен с 10000 до 20000


ПАРАМЕТРЫ ФИЛЬТРА:
  Частота дискретизации: 500 Гц
  Частота среза: 100 Гц
  Заданный порядок фильтра: 10000
  Эффективный порядок после самосвертки: 20000
  Тип окна: hamming
  Нормированная частота среза: 0.4000
  Сумма коэффициентов фильтра: 1.000000 (должна быть ≈ 1.0)
  Самосвертка импульсной характеристики: ДА
  Улучшение: более крутой склон АЧХ, лучшая подавление в полосе задерживания

ПАРАМЕТРЫ СИГНАЛА:
  Длина сигнала: 2000 отсчетов
  Длительность сигнала: 4.00 секунд
  Компоненты: ['50 Гц (A=1.0)', '150 Гц (A=0.5)', '300 Гц (A=0.3)']
  Частота Найквиста: 250.0 Гц

